# The grstbx viewer in a JupyterHub session

`grstbx-viewer` is a [Panel](https://panel.holoviz.org) application: a Python process reads the GRS
products and computes the images, spectra and time series requested by the browser. In a JupyterHub
session, this process runs on the machine of the hub, next to the data, and only the images reach your
browser. There are three ways to open it:

| | how | needs | best for |
|---|---|---|---|
| 1 | **inline** in a notebook cell, {func}`grstbx.app.panel` | nothing more | quick look, next to your analysis |
| 2 | **browser tab** started from the notebook, through the hub proxy | `jupyter-server-proxy` | full-page dashboard while you keep using the notebook |
| 3 | **browser tab** started from a JupyterHub terminal, `grstbx-viewer` | `jupyter-server-proxy` | long sessions, independent of any kernel |

In all cases the products are read on the hub, with the memory of your JupyterHub session: each
dashboard keeps up to `cache_size` (2 GB by default) of data in cache.

## Setup

Products to open (path, glob pattern or folder of the GRS batches, see {doc}`../dashboard`) and time series options. Here, the L2B products of tile 31TGJ (Lac de
Sainte-Croix) for one month:

In [ ]:
import os
import socket

import panel as pn
from IPython.display import HTML, display

from grstbx import app

PRODUCTS = '/data/satellite/Sentinel-2/L2B_zarr/31TGJ'
OPTIONS = dict(tile='31TGJ', start='2025-06-01', end='2025-06-30', basemap='CARTO Positron')

pn.extension()

JupyterHub describes the session in environment variables: `JUPYTERHUB_SERVICE_PREFIX` is the base
of the URLs of your server (`/user/<name>/`), from which the proxy URLs are built. Check that you are in
a JupyterHub session and that `jupyter-server-proxy` (needed by methods 2 and 3) is installed:

In [ ]:
prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX')
print('JupyterHub user  :', os.environ.get('JUPYTERHUB_USER', '- (not in a JupyterHub session)'))
print('URL prefix       :', prefix or '-')
try:
    import jupyter_server_proxy  # noqa: F401
    print('server proxy     : installed')
except ImportError:
    print('server proxy     : missing, pip install jupyter-server-proxy and restart your server')

## 1. Inline in the notebook

{func}`grstbx.app.panel` displays the dashboard in the output of the cell: the sidebar on the left of the
map. It is served by the kernel of the notebook through the Jupyter connection itself, so it works on any
hub without configuration. It takes the arguments of `grstbx-viewer` (see {func}`~grstbx.app.create_app`):
leave `PRODUCTS` out to choose them in the *File browser* of the sidebar (`root=` sets its folder).

In [ ]:
app.panel(PRODUCTS, width=900, height=650, **OPTIONS)

The dashboard lives as long as the kernel: restarting or shutting down the kernel closes it. The
outputs are not saved with the notebook (they are interactive), so run the cell again after reopening it.

## 2. In a browser tab, from the notebook

`jupyter-server-proxy` forwards `https://<hub>/user/<name>/proxy/<port>/` to a server listening on
`<port>` on the machine of your session, after the hub has authenticated you. Start the dashboard server
in a thread of the kernel (the notebook stays usable), on a free port, listening on `127.0.0.1` only, so
that it is reached through the authenticated proxy and not directly from the network:

In [ ]:
with socket.socket() as s:            # free port: other users of the machine may use 5006
    s.bind(('127.0.0.1', 0))
    PORT = s.getsockname()[1]

server = pn.serve(lambda: app.create_app(PRODUCTS, **OPTIONS), port=PORT, address='127.0.0.1',
                  show=False, threaded=True, title='grstbx viewer',
                  websocket_origin='*')   # the page is served under the hub's domain, see below

url = f'{prefix}proxy/{PORT}/' if prefix else f'http://localhost:{PORT}/'   # outside JupyterHub: local server
display(HTML(f'<a href="{url}" target="_blank">Open the grstbx viewer</a> ({url})'))

Click the link: the dashboard opens in a new tab, each tab opening its own session. The browser loads
the page from the hub's domain, so the server accepts any websocket origin (`websocket_origin='*'`):
this is safe because it listens on `127.0.0.1` only and is reached through the hub proxy, which checks
your login. Stop the server when you are done (it also stops with the kernel):

In [ ]:
server.stop()

## 3. In a browser tab, from a terminal

To keep the dashboard running independently of any notebook, start `grstbx-viewer` in a JupyterHub
terminal (*File > New > Terminal*), with the same options as above:

```console
grstbx-viewer /data/satellite/Sentinel-2/L2B_zarr/31TGJ --tile 31TGJ --start 2025-06-01 --end 2025-06-30 \
              --no-show --address 127.0.0.1 --port 5006 --allow-websocket-origin <hub domain>
```

- `--no-show`: there is no web browser on the hub;
- `--address 127.0.0.1`: reached only through the proxy;
- `--allow-websocket-origin`: the domain of the hub as it appears in the address bar of your browser
  (e.g. `jupyterhub.mylab.fr`), otherwise the page stays empty;
- `--port`: choose another one if 5006 is taken (*Address already in use*).

Then open `https://<hub domain>/user/<name>/proxy/5006/`, and stop the dashboard with `Ctrl+C` in the
terminal. Run it under `nohup` or in a `tmux` session to keep it after closing the terminal tab, for as
long as your JupyterHub server runs.

## Troubleshooting

| symptom | cause and remedy |
|---|---|
| 404 on the `/proxy/<port>/` URL | `jupyter-server-proxy` is not installed in the environment of the Jupyter server: install it and restart your server (*File > Hub Control Panel*) |
| *502 Bad Gateway* | no server on that port: it was stopped, or started on another port |
| empty page, *Refusing websocket connection* in the logs | websocket origin not allowed: `websocket_origin='*'` (method 2) or `--allow-websocket-origin <hub domain>` (method 3) |
| inline dashboard not displayed | `pn.extension()` not run, or a kernel without the Panel / Bokeh extensions; run the setup cell again |
| no basemap | the Stadia Maps styles need an API key outside `localhost` (`--stadia-api-key`); use the CARTO, OpenStreetMap or Esri basemaps |
| kernel killed, session slow | memory of the JupyterHub session exceeded: lower `cache_size` (`--cache-size` in GB), open fewer dates or a smaller period |